# STAT 8810 Project — Part 3: How do the results change with sample size?

This notebook collects the results of `02_fit_propensity.ipynb` and `03_ate_ipw.ipynb` run at
$n = 1000$ (`results/`), $n = 2000$ (`results_n2000/`) and $n = 5000$ (`results_n5000/`), and asks:

1. **Part 1:** does the propensity score MSE of each model shrink with $n$, and does the ranking of the models change?
2. **Part 2:** is the ATE bias of each method a **finite-sample** problem (shrinks as $n$ grows) or **systematic** (stays roughly constant)?
3. Does the bias track **balance on the nonlinear terms** of the outcome model rather than the PS MSE?

Everything else (data-generating process, hyperparameters, $R = 50$ replications, 5-fold cross-fitting)
is identical across $n$; the hyperparameters were chosen once on pilot data at $n = 1000$ and are **not** re-tuned.

**How to use:** run `02` and then `03` with `N = 2000`, then again with `N = 5000` (the `N` setting is
in the first code cell of each). Then run this notebook. Sample sizes whose results are missing are skipped.

Outputs go to `results_compare_n/`.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

TRUE_ATE = 1.5
N_ALL    = [1000, 2000, 5000]
OUT_DIR  = Path("results_compare_n")
OUT_DIR.mkdir(exist_ok=True)

def res_dir(n):
    return Path("results") if n == 1000 else Path(f"results_n{n}")

NEEDED = ["part1_metrics.csv", "part2_estimates.csv", "part2_balance_weights.csv"]
N_LIST = [n for n in N_ALL if all((res_dir(n) / f).exists() for f in NEEDED)]
print("sample sizes with complete results:", N_LIST)
for n in N_ALL:
    if n not in N_LIST:
        print(f"  n = {n}: missing -> run 02 and 03 with N = {n}")

PS_SCENARIOS = ["moderate", "strong"]
PS_MODELS    = ["logistic", "mlp", "gbm"]
METHODS      = ["naive", "logistic", "mlp", "gbm", "oracle"]
LABEL        = {"naive": "Naive", "logistic": "Logistic", "mlp": "MLP",
                "gbm": "Boosting", "oracle": "Oracle (true e)"}
COLOR        = {"naive": "grey", "logistic": "C0", "mlp": "C1", "gbm": "C2", "oracle": "k"}

part1 = pd.concat([pd.read_csv(res_dir(n) / "part1_metrics.csv").assign(n=n) for n in N_LIST])
est   = pd.concat([pd.read_csv(res_dir(n) / "part2_estimates.csv").assign(n=n) for n in N_LIST])
bal   = pd.concat([pd.read_csv(res_dir(n) / "part2_balance_weights.csv").assign(n=n) for n in N_LIST])
print("replications per (n, scenario):",
      est[est.method == "oracle"].groupby(["n", "ps"]).size().to_dict())

## 1. Part 1 — propensity score MSE by sample size

Mean (SD) of the out-of-fold MSE $\frac1n\sum_i(\hat e_i - e_i)^2$ over the replications, and how often
each flexible model beats logistic regression on the same dataset.

For a correctly specified parametric model the MSE falls roughly like $1/n$. Logistic regression is
misspecified here, so its MSE should level off at the approximation error; the flexible models should keep improving.

In [ ]:
p1 = (part1.groupby(["ps", "model", "n"], sort=False).mse.agg(["mean", "std"])
      .unstack("n"))
p1 = p1.reindex(pd.MultiIndex.from_product([PS_SCENARIOS, PS_MODELS]))
p1.index = pd.MultiIndex.from_tuples([(ps, LABEL[m]) for ps, m in p1.index], names=["ps", "model"])
p1.to_csv(OUT_DIR / "part1_mse_by_n.csv")
p1.round(4)

In [ ]:
wins = []
for (ps, n), g in part1.groupby(["ps", "n"]):
    w = g.pivot(index="rep", columns="model", values="mse")
    R = len(w)
    for a, b in [("mlp", "logistic"), ("gbm", "logistic"), ("mlp", "gbm")]:
        d = w[a] - w[b]
        wins.append(dict(ps=ps, n=n, comparison=f"{LABEL[a]} - {LABEL[b]}",
                         mean_mse_diff=d.mean(), se=d.std() / np.sqrt(R),
                         wins=f"{(d < 0).sum()}/{R}"))
wins = pd.DataFrame(wins).sort_values(["ps", "comparison", "n"])
wins.to_csv(OUT_DIR / "part1_paired_by_n.csv", index=False)
wins.round(5)

## 2. Part 2 — ATE bias, SD and RMSE by sample size

Main estimator: **Hájek** IPW (HT in the second table). `bias_over_mcse` is bias / MC SE: values beyond about ±2
mean the bias is real rather than Monte Carlo noise. `bias_share_mse` $=\text{bias}^2/\text{RMSE}^2$ is the share of the
error that comes from bias; if it grows with $n$, the bias does not shrink as fast as the SD and will dominate in large samples.

In [ ]:
def summarize_by_n(col):
    out = []
    for (ps, m, n), g in est.groupby(["ps", "method", "n"], sort=False):
        v = g[col].dropna()
        if len(v) == 0:
            continue
        bias, sd = v.mean() - TRUE_ATE, v.std()
        rmse = np.sqrt(np.mean((v - TRUE_ATE)**2))
        out.append(dict(ps=ps, method=m, n=n, bias=bias, mc_se=sd / np.sqrt(len(v)),
                        bias_over_mcse=bias / (sd / np.sqrt(len(v))),
                        rel_bias_pct=bias / TRUE_ATE * 100, sd=sd, rmse=rmse,
                        bias_share_mse=bias**2 / rmse**2, sqrt_n_rmse=np.sqrt(n) * rmse))
    s = pd.DataFrame(out)
    s["order"] = s.method.map({m: i for i, m in enumerate(METHODS)})
    return s.sort_values(["ps", "order", "n"]).drop(columns="order")

summ_hj = summarize_by_n("hajek")
summ_ht = summarize_by_n("ht")
summ_hj.to_csv(OUT_DIR / "part2_summary_hajek_by_n.csv", index=False)
summ_ht.to_csv(OUT_DIR / "part2_summary_ht_by_n.csv", index=False)
print("Hajek IPW")
summ_hj.assign(method=summ_hj.method.map(LABEL)).round(4)

In [ ]:
print("Horvitz-Thompson IPW")
summ_ht.assign(method=summ_ht.method.map(LABEL)).round(4)

**Compact view (Hájek):** bias and RMSE side by side for each $n$.

In [ ]:
compact = (summ_hj[summ_hj.method != "naive"]
           .assign(method=lambda d: d.method.map(LABEL))
           .pivot_table(index=["ps", "method"], columns="n", values=["bias", "rmse"], sort=False))
compact.round(3)

**Paired comparison of ATE errors (Hájek).** Mean of the difference in squared error to the true ATE, replication by replication,
and how often the first method is closer to the truth.

In [ ]:
pair_rows = []
for (ps, n), g in est.groupby(["ps", "n"]):
    w = g.pivot(index="rep", columns="method", values="hajek")
    R = len(w)
    err = (w - TRUE_ATE)**2
    for a, b in [("mlp", "logistic"), ("gbm", "logistic"), ("mlp", "gbm")]:
        d = err[a] - err[b]
        pair_rows.append(dict(ps=ps, n=n, comparison=f"{LABEL[a]} vs {LABEL[b]}",
                              mean_sq_err_diff=d.mean(), se=d.std() / np.sqrt(R),
                              closer=f"{(np.abs(w[a] - TRUE_ATE) < np.abs(w[b] - TRUE_ATE)).sum()}/{R}"))
pairs = pd.DataFrame(pair_rows).sort_values(["ps", "comparison", "n"])
pairs.to_csv(OUT_DIR / "part2_paired_by_n.csv", index=False)
pairs.round(4)

## 3. Balance and weights by sample size

Mean over replications of the largest |SMD| after weighting, by covariate group, plus the median maximum weight
and the effective sample size of the treated arm as a share of the treated units (about $n/2$).

In [ ]:
bal_tab = bal.copy()
bal_tab["ess_treated_share"] = bal_tab.ess_treated / (bal_tab.n / 2)
bal_tab = (bal_tab[bal_tab.method != "Naive"]
           .set_index(["ps", "method", "n"])
           [["maxsmd_confounders", "maxsmd_treatment_only", "maxsmd_nonlinear_terms",
             "max_weight_median", "ess_treated_share"]]
           .unstack("n"))
bal_tab.to_csv(OUT_DIR / "balance_by_n.csv")
bal_tab.round(3)

## 4. Figures

In [ ]:
# (a) Part 1: PS MSE vs n (log-log), mean +- 2 SE over replications
fig, axes = plt.subplots(1, 2, figsize=(11, 4.2), sharey=True)
for ax, ps in zip(axes, PS_SCENARIOS):
    for m in PS_MODELS:
        g = part1[(part1.ps == ps) & (part1.model == m)].groupby("n").mse
        mu, se = g.mean(), g.std() / np.sqrt(g.size())
        ax.errorbar(mu.index, mu.values, yerr=2 * se.values, marker="o", capsize=3,
                    color=COLOR[m], label=LABEL[m])
    ax.set_xscale("log"); ax.set_yscale("log")
    ax.set_xticks(N_LIST, [str(n) for n in N_LIST]); ax.minorticks_off()
    ax.set_xlim(N_LIST[0] * 0.8, N_LIST[-1] * 1.25)
    ax.set_xlabel("sample size n"); ax.set_title(f"PS = {ps}")
    ax.legend(fontsize=8)
axes[0].set_ylabel("PS MSE vs true e (log scale)")
fig.suptitle("Part 1: propensity score MSE by sample size")
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_n_part1_mse.png", dpi=150)
plt.show()

In [ ]:
# (b) Part 2: Hajek bias (+- 2 MC SE) and RMSE vs n
meths = [m for m in METHODS if m != "naive"]       # naive is off the scale; see the table
fig, axes = plt.subplots(2, 2, figsize=(11, 7.5), sharex=True)
for j, ps in enumerate(PS_SCENARIOS):
    s = summ_hj[summ_hj.ps == ps]
    for k, m in enumerate(meths):
        g = s[s.method == m]
        jitter = 1 + 0.02 * (k - 2)                 # small horizontal offset so error bars don't overlap
        axes[0, j].errorbar(g.n * jitter, g.bias, yerr=2 * g.mc_se, marker="o", capsize=3,
                            color=COLOR[m], label=LABEL[m])
        axes[1, j].plot(g.n, g.rmse, marker="o", color=COLOR[m], label=LABEL[m])
    # reference: the oracle RMSE at the smallest n, shrinking like 1/sqrt(n)
    o = s[s.method == "oracle"].set_index("n").rmse
    nn = np.array(N_LIST, float)
    axes[1, j].plot(nn, o.loc[N_LIST[0]] * np.sqrt(N_LIST[0] / nn), "k:", lw=1,
                    label=r"$\propto 1/\sqrt{n}$")
    axes[0, j].axhline(0, color="k", lw=0.8)
    axes[0, j].set_title(f"Bias (Hajek, ± 2 MC SE) — PS = {ps}")
    axes[1, j].set_title(f"RMSE (Hajek) — PS = {ps}")
    axes[1, j].set_ylim(bottom=0)
    for ax in axes[:, j]:
        ax.set_xscale("log")
        ax.set_xticks(N_LIST, [str(n) for n in N_LIST]); ax.minorticks_off()
        ax.set_xlim(N_LIST[0] * 0.8, N_LIST[-1] * 1.25)
        ax.legend(fontsize=7)
    axes[1, j].set_xlabel("sample size n")
fig.suptitle("Part 2: ATE bias and RMSE by sample size (true ATE = 1.5)")
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_n_part2_bias_rmse.png", dpi=150)
plt.show()

In [ ]:
# (c) What drives the bias? Method-level points (one per scenario x method x n):
#     left: PS MSE vs |bias|; right: balance on the nonlinear outcome terms vs |bias|
pts = (est[est.method.isin(PS_MODELS)]
       .groupby(["ps", "method", "n"])
       .agg(ps_mse=("ps_mse", "mean"), maxsmd_nonlin=("maxsmd_nonlin", "mean"),
            bias=("hajek", lambda v: v.mean() - TRUE_ATE))
       .reset_index())
mk = {"moderate": "o", "strong": "^"}
size = {n: s for n, s in zip(N_ALL, [30, 60, 110])}
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.5), sharey=True)
for ax, xcol, xlab in [(axes[0], "ps_mse", "mean PS MSE"),
                       (axes[1], "maxsmd_nonlin", "mean max |SMD|, nonlinear outcome terms")]:
    for _, r in pts.iterrows():
        ax.scatter(r[xcol], abs(r.bias), marker=mk[r.ps], s=size[r.n], color=COLOR[r.method], alpha=.8)
    ax.set_xlabel(xlab)
axes[0].set_ylabel("|bias| of Hajek ATE")
handles = ([plt.Line2D([], [], color=COLOR[m], marker="s", ls="", label=LABEL[m]) for m in PS_MODELS] +
           [plt.Line2D([], [], color="grey", marker=mk[p], ls="", label=f"PS = {p}") for p in PS_SCENARIOS] +
           [plt.scatter([], [], s=size[n], color="grey", label=f"n = {n}") for n in N_LIST])
axes[1].legend(handles=handles, fontsize=7, loc="upper left", bbox_to_anchor=(1.02, 1))
fig.suptitle("Does the ATE bias follow the PS MSE, or the balance on terms that matter for the outcome?")
fig.tight_layout()
fig.savefig(OUT_DIR / "fig_n_bias_drivers.png", dpi=150)
plt.show()

for xcol in ["ps_mse", "maxsmd_nonlin"]:
    rho = pts[xcol].rank().corr(pts.bias.abs().rank())
    print(f"Spearman correlation of {xcol} with |bias| over the {len(pts)} points: {rho:.2f}")

## 5. How to read the results

- **Bias of a flexible model shrinks clearly with $n$** (and `bias_share_mse` does not grow): the bias at $n = 1000$ is mainly a
  finite-sample approximation error — the model needs more data to learn the nonlinear propensity score.
- **Bias roughly constant in $n$** while the SD shrinks: the bias is systematic (for logistic regression this is the expected
  consequence of misspecification); RMSE will level off instead of falling like $1/\sqrt n$.
- **Oracle:** unbiased at every $n$, RMSE close to the $1/\sqrt n$ reference line.
- **Ranking changes with $n$:** e.g. the flexible models overtaking logistic regression only at larger $n$ is itself a finding worth reporting.